In [5]:
!pip install catboost joblib pandas scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# HCC PRIORITY - SVM
# ============================================================

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(
    ""F:\Risk Adjustment and HCC suspecting Assistant\suspects_clean_ready_for_augmentation_with_severity.csv""
)

df = df.dropna(
    how="all"
).reset_index(drop=True)


# ============================================================
# TARGET
# ============================================================

df["disease_priority"] = (
    df["disease_priority"]
    .astype(str)
    .str.upper() # type: ignore
    .str.strip()
)


df = df[
    df["disease_priority"].isin(
        ["LOW", "MEDIUM", "HIGH"]
    )
].copy()


target_map = {
    "LOW": 0,
    "MEDIUM": 1,
    "HIGH": 2
}


df["target"] = (
    df["disease_priority"]
    .map(target_map)
)


# ============================================================
# FEATURES
# ============================================================

FEATURES = [

    "hcc_v28",
    "gap_type",
    "suspect_type",
    "status",
    "latest_context",

    "diagnosis_count",
    "unique_claim_count",
    "unique_event_count",
    "distinct_evidence_dates",
    "distinct_evidence_months",
    "distinct_sources",
    "principal_diagnosis_count",
    "prescription_support_count"

]

FEATURES = [
    x for x in FEATURES
    if x in df.columns
]


# ============================================================
# PATIENT SPLIT
# ============================================================

patient_target = (
    df.groupby("bene_id")["target"]
    .agg(lambda x: x.mode()[0])
)

patients = patient_target.index


train_patients, temp_patients = train_test_split(

    patients,

    test_size=0.30,

    random_state=42,

    stratify=patient_target.values

)


val_patients, test_patients = train_test_split(

    temp_patients,

    test_size=0.50,

    random_state=42,

    stratify=patient_target.loc[
        temp_patients
    ].values

)


train_patients = set(train_patients)
val_patients = set(val_patients)
test_patients = set(test_patients)


# ============================================================
# DATA
# ============================================================

train_df = df[
    df["bene_id"].isin(train_patients)
]

val_df = df[
    df["bene_id"].isin(val_patients)
]

test_df = df[
    df["bene_id"].isin(test_patients)
]


X_train = train_df[FEATURES]
X_val = val_df[FEATURES]
X_test = test_df[FEATURES]

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]


# ============================================================
# COLUMNS
# ============================================================

categorical_features = [

    col for col in FEATURES

    if X_train[col].dtype == "object"

]


numeric_features = [

    col for col in FEATURES

    if col not in categorical_features

]


# ============================================================
# PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",
        StandardScaler()
    )

])


categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )

])


preprocessor = ColumnTransformer([

    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),

    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )

])


# ============================================================
# SVM
# ============================================================

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",

        SVC(

            C=2.0,

            kernel="rbf",

            gamma="scale",

            class_weight="balanced",

            probability=True,

            random_state=42

        )

    )

])


# ============================================================
# TRAIN
# ============================================================

model.fit(
    X_train,
    y_train
)


# ============================================================
# PREDICT
# ============================================================

train_pred = model.predict(
    X_train
)

val_pred = model.predict(
    X_val
)

test_pred = model.predict(
    X_test
)


# ============================================================
# RESULTS
# ============================================================

train_acc = accuracy_score(
    y_train,
    train_pred
)

val_acc = accuracy_score(
    y_val,
    val_pred
)

test_acc = accuracy_score(
    y_test,
    test_pred
)

macro_f1 = f1_score(
    y_test,
    test_pred,
    average="macro",
    zero_division=0
)

balanced_acc = balanced_accuracy_score(
    y_test,
    test_pred
)


print("\n" + "=" * 70)
print("SVM RESULTS")
print("=" * 70)

print(
    f"Train Accuracy      : {train_acc:.4f}"
)

print(
    f"Validation Accuracy : {val_acc:.4f}"
)

print(
    f"Test Accuracy       : {test_acc:.4f}"
)

print(
    f"Generalization Gap  : "
    f"{train_acc - test_acc:.4f}"
)

print(
    f"Balanced Accuracy   : {balanced_acc:.4f}"
)

print(
    f"Macro F1            : {macro_f1:.4f}"
)


print(
    classification_report(

        y_test,

        test_pred,

        target_names=[
            "LOW",
            "MEDIUM",
            "HIGH"
        ],

        zero_division=0

    )
)


print(
    confusion_matrix(
        y_test,
        test_pred
    )
)


# ============================================================
# SAVE
# ============================================================

joblib.dump(
    model,
    "hcc_svm_model.pkl"
)

print(
    "\nModel saved as hcc_svm_model.pkl"
)

FileNotFoundError: [Errno 2] No such file or directory: 'hcc_augmented_3500_full_dataset.csv'